In [0]:
-- High Level Sales Analysis

-- 1: What was the total quantity sold for all products

-- total sold
SELECT
sum(qty) as total_sold
FROM sales;

-- total sold by product
SELECT pd.product_name,
SUM(s.qty) as total_sold
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name
ORDER BY SUM(s.qty) desc;

-- 2: What was the total generated revenue for all products before discounts

-- total revenue
SELECT
sum(qty * price) as total_rev
FROM sales;

-- total revenue by product
SELECT pd.product_name,
SUM(s.qty * s.price) as rev_gen
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name
ORDER BY SUM(s.qty * s.price) DESC;

-- 3: What was the total discount amount for all products

-- total discount
SELECT
SUM(qty * price * discount) / 100 as total_discount
FROM sales;

-- total discount by product
SELECT
pd.product_name,
SUM(s.qty * s.price * s.discount) / 100 as total_discount
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name
ORDER BY (SUM(s.qty * s.price * s.discount) / 100) desc;




-- Transaction Analysis

-- 1: How many unique transactions were there?
SELECT COUNT(DISTINCT txn_id) as unique_transaction_ids FROM sales;

-- 2: What is the average unique products purchased in each transaction?
WITH cte as (SELECT count(distinct prod_id) as uniq_prods_purchased FROM sales
GROUP BY txn_id)
SELECT ROUND(AVG(uniq_prods_purchased), 0) as average_uniq_prods FROM cte;

-- 3: What are the 25th, 50th and 75th percentile values for the revenue per transaction?
WITH cte as (SELECT txn_id,
SUM(qty * price * ((100 - discount) / 100.0)) as total_revenue
FROM sales
GROUP BY txn_id)

SELECT distinct percentile_cont(0.25) WITHIN GROUP (ORDER BY total_revenue) OVER () as p25,
percentile_cont(0.5) WITHIN GROUP (ORDER BY total_revenue) OVER () as p50,
percentile_cont(0.75) WITHIN GROUP (ORDER BY total_revenue) OVER () as p75
FROM cte;

-- 4: What is the average discount value per transaction?

SELECT SUM(qty * price * (cast(discount as decimal(10,2)) /100) ) 
/ COUNT (distinct txn_id) AS average_discount 
FROM sales;

-- 5: What is the percentage split of all transactions for members vs non-members?
WITH txn_membership AS (
  SELECT
    txn_id,
    MAX(member) AS member_flag    
  FROM sales
  GROUP BY txn_id
)
SELECT
  ROUND(SUM(CASE WHEN member_flag = true  THEN 1 END) * 100.0 / COUNT(*), 2) AS member_percentage,
  ROUND(SUM(CASE WHEN member_flag = false THEN 1 END) * 100.0 / COUNT(*), 2) AS non_member_percentage
FROM txn_membership;

-- 6: What is the average revenue for member transactions and non-member transactions?
WITH cte as (SELECT
distinct txn_id,
member,
SUM(price * qty * ((100 - discount) / 100)) as revenue
FROM sales
GROUP BY txn_id, member)

SELECT distinct member,
ROUND(AVG(revenue) OVER (PARTITION BY member), 2) as avg_revenue
FROM cte;


-- Product Analysis

-- 1: What are the top 3 products by total revenue before discount?

SELECT pd.product_name,
SUM(s.price * s.qty) as total_revenue
FROM sales s
JOIN product_details pd ON 
pd.product_id = s.prod_id
GROUP BY pd.product_name
ORDER BY total_revenue DESC
LIMIT 3;

-- 2: What is the total quantity, revenue and discount for each segment?

SELECT 
pd.segment_name,
SUM(s.qty) as total_quantity,
SUM(s.qty * pd.price) as total_revenue,
ROUND(SUM(s.qty * pd.price * (s.discount / 100)), 2) as total_discount
FROM sales s
JOIN product_details pd
ON pd.product_id = s.prod_id
GROUP BY pd.segment_name;

-- 3: What is the top selling product for each segment?

WITH rankings as (SELECT 
pd.product_name,
pd.segment_name,
SUM(s.qty * s.price) as total_revenue,
ROW_NUMBER() OVER (PARTITION BY pd.segment_name ORDER BY SUM(s.qty * s.price) DESC) as rank
FROM sales s
JOIN product_details pd
ON pd.product_id = s.prod_id
GROUP BY pd.segment_name, pd.product_name)

SELECT * FROM rankings 
WHERE rank = 1;

-- 4: What is the total quantity, revenue and discount for each category?

SELECT
pd.category_name,
SUM(s.qty) as total_qty,
SUM(s.qty * s.price) as total_rev,
ROUND(SUM(s.qty * s.price * s.discount / 100.0), 2) as total_discount
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.category_name
ORDER BY total_rev DESC;

-- 5: What is the top selling product for each category?

WITH rankings as (SELECT 
pd.product_name,
pd.category_name,
SUM(s.qty * s.price) as total_revenue,
ROW_NUMBER() OVER (PARTITION BY pd.category_name ORDER BY SUM(s.qty * s.price) DESC) as rank
FROM sales s
JOIN product_details pd
ON pd.product_id = s.prod_id
GROUP BY pd.category_name, pd.product_name)

SELECT * FROM rankings 
WHERE rank = 1;

-- 6: What is the percentage split of revenue by product for each segment?

WITH total_rev as (SELECT 
pd.product_name,
pd.segment_name,
SUM(s.qty * s.price) as revenue
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.product_name, pd.segment_name)

SELECT product_name,
segment_name,
SUM(revenue) OVER (PARTITION BY segment_name) as total_segment_revenue,
ROUND(revenue / SUM(revenue) OVER (PARTITION BY segment_name) * 100.0, 2) as pct_segment_revenue
FROM total_rev;

-- 7: What is the percentage split of revenue by segment for each category?
WITH total_rev as (SELECT 
pd.segment_name,
pd.category_name,
SUM(s.qty * s.price) as revenue
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.segment_name, pd.category_name)

SELECT segment_name,
category_name,
SUM(revenue) OVER (PARTITION BY category_name) as total_category_revenue,
ROUND(revenue / SUM(revenue) OVER (PARTITION BY category_name) * 100.0, 2) as pct_category_revenue
FROM total_rev;

-- 8: What is the percentage split of total revenue by category?

WITH totals as (SELECT 
pd.category_name,
SUM(s.qty * s.price) as total_revenue
FROM sales s
JOIN product_details pd
ON s.prod_id = pd.product_id
GROUP BY pd.category_name)

SELECT category_name,
ROUND(100.0 * total_revenue / SUM(total_revenue) OVER(), 2) as pct_revenue
FROM totals;

-- 9: What is the total transaction “penetration” for each product? (hint: penetration = number of transactions where at least 1 quantity of a product was purchased divided by total number of transactions)

WITH cte as (SELECT prod_id,
count(distinct txn_id) as total_transactions
FROM sales
GROUP BY prod_id),

total_unq_trans as (SELECT 
count(distinct txn_id) as total_transactions_all
FROM sales)

SELECT cte.prod_id,
cte.total_transactions,
total_unq_trans.total_transactions_all,
ROUND(cte.total_transactions * 1.0 /total_unq_trans.total_transactions_all, 2) as penetration
FROM cte, total_unq_trans

